# Deep Learning: perché la profondità conta

Il codice del capitolo [«Deep Learning: perché la profondità conta»](https://book.paithon.it/main/DeepLearning/overview.html), *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro: [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Su Colab quasi tutto c'è già; questa riga serve altrove.
%pip install -q numpy torch torchvision

In [ ]:
# Mostra il valore di ogni riga, come i commenti «# ->» del libro.
try:
    from IPython.core.interactiveshell import InteractiveShell
    InteractiveShell.ast_node_interactivity = 'all'
except ImportError:      # fuori da IPython non serve e non c'è
    pass

## Deep Learning: perché la profondità conta

[Leggi la pagina](https://book.paithon.it/main/DeepLearning/overview.html)


### Quante regioni taglia una rete


In [ ]:
# Un ingresso, un'uscita, K strati nascosti da D neuroni ReLU ciascuno.
def parametri(D, K):
    return 3 * D + 1 + (K - 1) * D * (D + 1)

def regioni(D, K):        # quante ne puo' fare al massimo
    return (D + 1) ** K

bilancio = parametri(10, 5)
print(f"profonda: K=5, D=10 -> {bilancio} parametri, {regioni(10, 5)} regioni")

# La rete a uno strato piu' piccola che quel bilancio se lo puo' permettere.
D = next(d for d in range(1, 10_000) if parametri(d, 1) >= bilancio)
print(f"piatta:   K=1, D={D} -> {parametri(D, 1)} parametri, "
      f"{regioni(D, 1)} regioni")
print(f"rapporto: {regioni(10, 5) / regioni(D, 1):.0f} volte")

### La pila, in codice


In [ ]:
from torch import nn

# in ingresso: un gruppo di immagini a colori, alte e larghe 128 pixel
model = nn.Sequential(
    nn.Conv2d(3, 32, 3), nn.ReLU(),    # primi strati: bordi e linee
    nn.MaxPool2d(2),
    nn.Conv2d(32, 64, 3), nn.ReLU(),   # strati intermedi: texture e parti
    nn.MaxPool2d(2),
    nn.Conv2d(64, 128, 3), nn.ReLU(),  # parti più grandi, oggetti
    nn.AdaptiveAvgPool2d(1),           # media di ogni foglio di risultati
    nn.Flatten(),
    nn.Linear(128, 10),                # la classe finale (un punteggio per classe)
)

## Reti convoluzionali (CNN)

[Leggi la pagina](https://book.paithon.it/main/DeepLearning/reti-convoluzionali.html)


### Il pooling: mappe più piccole, e cosa si guadagna


In [ ]:
import torch
from torch.nn.functional import max_pool2d

def dopo_lo_spostamento(mappa):
    """La stessa mappa letta due volte, sfalsata di un pixel, dopo il pooling."""
    return max_pool2d(mappa[..., :-1], 2), max_pool2d(mappa[..., 1:], 2)

lato = 16
uguali = 0
for riga in range(lato):                 # un picco isolato, una posizione per volta
    for colonna in range(lato):
        m = torch.zeros(1, 1, lato, lato + 1)
        m[0, 0, riga, colonna] = 1.0
        a, b = dopo_lo_spostamento(m)
        uguali += int(torch.equal(a, b))
print(f"picco isolato: identica {uguali} volte su {lato * lato}")

g = torch.Generator().manual_seed(0)     # una mappa fitta di valori tutti diversi
uguali = sum(int(torch.equal(*dopo_lo_spostamento(
                 torch.rand(1, 1, lato, lato + 1, generator=g))))
             for _ in range(200))
print(f"mappa densa:   identica {uguali} volte su 200")

### L'architettura tipica


In [ ]:
import torch
from torch import nn

model = nn.Sequential(
    # blocco 1: 32 filtri 3x3, mappe grandi come l'input
    nn.Conv2d(1, 32, 3, padding="same"), nn.ReLU(),
    nn.MaxPool2d(2),
    # blocco 2: più filtri man mano che le mappe rimpiccioliscono.
    # Il primo numero è 32: ogni filtro di questo strato legge tutte
    # e 32 le mappe che escono dal blocco di sopra.
    nn.Conv2d(32, 64, 3, padding="same"), nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),                  # srotola la pila in una fila di numeri
    nn.Linear(64 * 7 * 7, 10),     # 10 classi (logit)
)

# quattro immagini in scala di grigi da 28x28: (immagini, canali, righe, colonne)
x = torch.zeros(4, 1, 28, 28)
for strato in model:
    x = strato(x)
    print(f"{strato.__class__.__name__:10s} -> {tuple(x.shape)}")

### Lo stesso conto con meno moltiplicazioni


In [ ]:
import numpy as np

# F(2,3): due uscite di un filtro da tre pesi con quattro moltiplicazioni
BT2 = np.array([[1, 0, -1, 0], [0, 1, 1, 0], [0, -1, 1, 0], [0, 1, 0, -1]])
G2 = np.array([[1, 0, 0], [1/2, 1/2, 1/2], [1/2, -1/2, 1/2], [0, 0, 1]])
AT2 = np.array([[1, 1, 1, 0], [0, 1, -1, -1]])
# F(4,3): quattro uscite con sei moltiplicazioni (le matrici di Lavin e Gray)
BT4 = np.array([[4, 0, -5, 0, 1, 0], [0, -4, -4, 1, 1, 0], [0, 4, -4, -1, 1, 0],
                [0, -2, -1, 2, 1, 0], [0, 2, -1, -2, 1, 0], [0, 4, 0, -5, 0, 1]])
G4 = np.array([[1/4, 0, 0], [-1/6, -1/6, -1/6], [-1/6, 1/6, -1/6],
               [1/24, 1/12, 1/6], [1/24, -1/12, 1/6], [0, 0, 1]])
AT4 = np.array([[1, 1, 1, 1, 1, 0], [0, 1, -1, 2, -2, 0],
                [0, 1, 1, 4, 4, 0], [0, 1, -1, 8, -8, 1]])

d, g = np.array([1., 2, 3, 4]), np.array([1., 2, 3])
print("diretta :", [float(d[i:i + 3] @ g) for i in range(2)], "con 6 moltiplicazioni")
print("Winograd:", [float(v) for v in AT2 @ ((G2 @ g) * (BT2 @ d))], "con 4 moltiplicazioni")
h = np.float16  # a 16 bit, oltre 2048 i numeri vanno di due in due
print("a 16 bit: 2048 + 1 + 1 =", float(h(2048) + h(1) + h(1)),
      "  1 + 1 + 2048 =", float(h(1) + h(1) + h(2048)))

# Da qui in poi solo operazioni elemento per elemento, in un ordine fissato:
# gli errori di arrotondamento non dipendono dal processore che fa i conti.
def per(M, X):
    """M @ X lungo il penultimo asse di X, sommando nell'ordine degli indici."""
    righe = []
    for i in range(M.shape[0]):
        somma = M[i, 0] * X[..., 0, :]
        for j in range(1, M.shape[1]):
            somma = somma + M[i, j] * X[..., j, :]
        righe.append(somma)
    return np.stack(righe, axis=-2)

def trasforma(M, X):                                  # M X M^T sugli ultimi due assi
    return np.swapaxes(per(M, np.swapaxes(per(M, X), -1, -2)), -1, -2)

def diretta(img, filt, tipo, per_canale=False):
    """Nove prodotti per canale; di default li somma tutti in una catena sola,
    con per_canale=True somma prima i nove di ogni canale e poi i canali."""
    img, filt = img.astype(tipo), filt.astype(tipo)
    C, H, W = img.shape
    uscita = np.zeros((H - 2, W - 2), tipo)
    for c in range(C):
        parziale = np.zeros((H - 2, W - 2), tipo) if per_canale else uscita
        for a in range(3):
            for b in range(3):
                parziale = parziale + img[c, a:a + H - 2, b:b + W - 2] * filt[c, a, b]
        uscita = uscita + parziale if per_canale else parziale
    return uscita

def winograd(img, filt, BT, G, AT, tipo):
    """Convoluzione 3x3 a tessere m x m: filtro e tessere trasformati, prodotto
    elemento per elemento sommato sui canali, trasformazione all'indietro."""
    BT, G, AT = BT.astype(tipo), G.astype(tipo), AT.astype(tipo)
    img, filt = img.astype(tipo), filt.astype(tipo)
    C, H, W = img.shape
    m, t = AT.shape[0], BT.shape[0]
    n = (H - 2) // m                                   # tessere per lato
    tessere = np.stack([np.stack([img[:, i * m:i * m + t, j * m:j * m + t]
                                  for j in range(n)], axis=1) for i in range(n)], axis=1)
    U = trasforma(G, filt)                             # una volta per filtro
    V = trasforma(BT, tessere)                         # una volta per tessera
    somma = U[0][None, None] * V[0]
    for c in range(1, C):                              # la somma sui canali
        somma = somma + U[c][None, None] * V[c]
    Y = trasforma(AT, somma)                           # n x n tessere di m x m uscite
    return Y.transpose(0, 2, 1, 3).reshape(n * m, n * m)

rng = np.random.default_rng(0)
img, filt = rng.normal(size=(64, 26, 26)), rng.normal(size=(64, 3, 3))  # 64 canali
esatta = diretta(img, filt, np.float64)
print(f"\n{'':14}{'moltipl. per uscita':>21}{'errore float32':>16}{'errore float16':>16}")
for nome, per_uscita, calcolo in [
        ("diretta", 9, lambda tipo: diretta(img, filt, tipo)),
        ("per canale", 9, lambda tipo: diretta(img, filt, tipo, per_canale=True)),
        ("F(2x2, 3x3)", 16 / 4, lambda tipo: winograd(img, filt, BT2, G2, AT2, tipo)),
        ("F(4x4, 3x3)", 36 / 16, lambda tipo: winograd(img, filt, BT4, G4, AT4, tipo))]:
    errori = [np.abs(calcolo(tipo) - esatta).max() / np.abs(esatta).max()
              for tipo in (np.float32, np.float16)]
    print(f"{nome:14}{per_uscita:21.2f}{errori[0]:16.1e}{errori[1]:16.1e}")

### Lo stesso conto con meno spostamenti


In [ ]:
import numpy as np

rng = np.random.default_rng(0)
R, H = 3, 6                                 # filtro R x R, immagine H x H
filtro = rng.integers(-2, 3, (R, R))
immagine = rng.integers(0, 5, (H, H))
E = H - R + 1                               # righe (e colonne) della mappa d'uscita

def riga_per_riga(riga_filtro, riga_immagine):
    """Convoluzione 1D: la riga del filtro scorre sulla riga dell'immagine."""
    return np.array([riga_filtro @ riga_immagine[x:x + R] for x in range(E)])

# il banco (i, j), contando da zero, tiene la riga i del filtro e riceve la riga
# i + j dell'immagine;
# la colonna j somma i suoi R risultati e dà la riga j dell'uscita
banchi = {(i, j): riga_per_riga(filtro[i], immagine[i + j]) for i in range(R) for j in range(E)}
uscita = np.array([sum(banchi[i, j] for i in range(R)) for j in range(E)])

diretta = np.array([[(filtro * immagine[y:y + R, x:x + R]).sum() for x in range(E)]
                    for y in range(E)])
print("uguale alla convoluzione diretta:", np.array_equal(uscita, diretta))

# chi usa che cosa: ogni riga del filtro, ogni riga dell'immagine, ogni riga dell'uscita
usi_filtro = [sum(1 for (i, j) in banchi if i == r) for r in range(R)]
usi_immagine = [sum(1 for (i, j) in banchi if i + j == h) for h in range(H)]
print(f"{R} x {E} banchi; ogni riga del filtro serve {usi_filtro} banchi,"
      f" le righe dell'immagine {usi_immagine}, ogni riga d'uscita somma {R} banchi")
moltiplicazioni = E * E * R * R
print(f"{moltiplicazioni} moltiplicazioni; letture se ogni numero arriva a ogni conto:"
      f" {2 * moltiplicazioni}, se ognuno arriva una volta sola: {R * R + H * H}")

## Far funzionare le reti profonde

[Leggi la pagina](https://book.paithon.it/main/DeepLearning/ottimizzazione-regolarizzazione.html)


### Partire col piede giusto: l'inizializzazione


In [ ]:
import statistics
import torch
from torch import nn

torch.set_num_threads(1)

def norma_primo_strato(ricetta, blocchi, seme):
    torch.manual_seed(seme)
    strati = []
    for _ in range(blocchi):
        lineare = nn.Linear(100, 100)
        if ricetta == "He":
            nn.init.kaiming_normal_(lineare.weight, nonlinearity="relu")
            nn.init.zeros_(lineare.bias)
        elif ricetta == "Glorot":
            nn.init.xavier_normal_(lineare.weight)
            nn.init.zeros_(lineare.bias)
        strati += [lineare, nn.ReLU()]
    rete = nn.Sequential(*strati)
    (rete(torch.randn(64, 100)) ** 2).mean().backward()
    return rete[0].weight.grad.norm().item()

for blocchi in (40, 60):
    for ricetta in ("He", "Glorot", "default"):
        norme = [norma_primo_strato(ricetta, blocchi, s) for s in range(5)]
        print(f"{blocchi} blocchi, {ricetta:<8} mediana {statistics.median(norme):.1e}"
              f"   (da {min(norme):.1e} a {max(norme):.1e})")

### Bersagli meno netti: il label smoothing


In [ ]:
import math

import numpy as np

K, eps, passo = 5, 0.1, 0.5

netto = np.zeros(K)
netto[0] = 1.0                   # gatto sì, tutto il resto no
morbido = np.full(K, eps / K)
morbido[0] += 1 - eps            # nove e due decimi al gatto, due decimi agli altri

def softmax(z):
    e = np.exp(z - z.max())
    return e / e.sum()

def scendi(q, tappe):
    """Discesa del gradiente sui soli punteggi grezzi: la correzione vale p - q."""
    z = np.zeros(K)
    for t in range(1, max(tappe) + 1):
        z -= passo * (softmax(z) - q)
        if t in tappe:
            p = softmax(z)
            yield t, z[0] - z[1], p[0]

tappe = (10**3, 10**4, 10**5)
for (t, dn, pn), (_, dm, pm) in zip(scendi(netto, tappe), scendi(morbido, tappe)):
    print(f"{t:>6} passi | netto: distacco {dn:5.2f}, al gatto il {pn:8.4%}"
          f" | morbido: distacco {dm:4.2f}, al gatto il {pm:.4%}")

print("distacco previsto per il morbido:",
      round(math.log((K * (1 - eps) + eps) / eps), 2))

### Scendere bene: gli optimizer moderni


In [ ]:
import torch

def dopo_40_passi(momentum, weight_decay=0.0, a_mano=False):
    p = torch.nn.Parameter(torch.tensor([1.0]))
    opt = torch.optim.SGD([p], lr=0.1, momentum=momentum, weight_decay=weight_decay)
    for _ in range(40):
        opt.zero_grad(); p.grad = torch.zeros_like(p)   # gradiente nullo
        opt.step()
        if a_mano:                # il decadimento vero, fuori dall'ottimizzatore
            with torch.no_grad(): p.mul_(1 - 0.1 * 0.05)    # 1 - eta*lambda
    return p.item()

print(f"weight_decay, senza momentum: {dopo_40_passi(0.0, weight_decay=0.05):.4f}")
print(f"a mano,       senza momentum: {dopo_40_passi(0.0, a_mano=True):.4f}")
print(f"a mano,       momentum 0,9:   {dopo_40_passi(0.9, a_mano=True):.4f}")
print(f"weight_decay, momentum 0,9:   {dopo_40_passi(0.9, weight_decay=0.05):.4f}")

In [ ]:
for Opt in (torch.optim.SGD, torch.optim.AdamW, torch.optim.Adam):
    p = torch.nn.Parameter(torch.tensor([1.0]))
    opt = Opt([p], lr=0.5, weight_decay=0.1)
    opt.zero_grad(); p.grad = torch.zeros_like(p); opt.step()
    print(f"{Opt.__name__:5s} dopo un passo: {p.item():.2f}")

### Regolare il passo nel tempo


*Frammento illustrativo: nel libro mostra la forma, qui non si esegue.*

```python

from torch import nn, optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)   # passo iniziale

# dimezza il learning rate quando la loss di validazione smette di scendere
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer,
                                                 factor=0.5, patience=3)

for epoca in range(50):        # un'«epoca» è una passata su tutti i dati
    addestra_una_epoca(model, train_loader, criterion, optimizer)
    loss_val = valuta(model, val_loader, criterion)   # loss di validazione
    scheduler.step(loss_val)                          # decide se ridurre il passo
```


### Quando la generalizzazione arriva tardi: il grokking


In [ ]:
import numpy as np

p = 97                       # le somme si fanno a orologio, su 97 ore
tavola = np.arange(p)[:, None] + np.arange(p)[None, :]      # a + b
vero = tavola % p

coppie = p * p
viste = int(0.3 * coppie)    # se ne mostra il 30% e si chiede il resto
print(f"coppie in tutto: {coppie}, viste: {viste}, mai viste: {coppie - viste}")
print(f"tirando a caso si indovina una volta su {p}: {100 / p:.2f}%")

def punteggi(frequenze):
    "s(c) = somma su k di cos(2 pi k (a + b - c) / p)"
    diff = (tavola[:, :, None] - np.arange(p)[None, None, :]) % p
    return sum(np.cos(2 * np.pi * k * diff / p) for k in frequenze)

for frequenze in ([1], [1, 2, 3, 4, 5]):
    s = punteggi(frequenze)
    ordinati = np.sort(s, axis=2)
    print(f"frequenze usate: {len(frequenze)}  ->  "
          f"{(s.argmax(axis=2) == vero).sum()}/{coppie} somme esatte, "
          f"margine minimo {(ordinati[..., -1] - ordinati[..., -2]).min():.4f}")

## Le architetture che hanno fatto la storia

[Leggi la pagina](https://book.paithon.it/main/DeepLearning/architetture-storiche.html)


### Separare lo spazio dai canali: la convoluzione che sta in un telefono


In [ ]:
import torch
import torch.nn as nn

C_IN, C_OUT, K, H, W = 64, 128, 3, 56, 56
x = torch.randn(1, C_IN, H, W)

# convoluzione ordinaria: ogni filtro guarda tutti i canali in una volta sola
ordinaria = nn.Conv2d(C_IN, C_OUT, K, padding=1, bias=False)

# separabile: prima la parte spaziale, un filtro per canale (groups=C_IN),
# poi la parte fra i canali, una 1x1 che li rimescola
separabile = nn.Sequential(
    nn.Conv2d(C_IN, C_IN, K, padding=1, groups=C_IN, bias=False),   # depthwise
    nn.Conv2d(C_IN, C_OUT, 1, bias=False),                          # pointwise
)

def parametri(m):
    return sum(p.numel() for p in m.parameters())

print("stessa forma in uscita:", ordinaria(x).shape == separabile(x).shape,
      tuple(separabile(x).shape))
print(f"parametri, ordinaria : {parametri(ordinaria):>8,}")
print(f"parametri, separabile: {parametri(separabile):>8,}")
print(f"risparmio            : {parametri(ordinaria) / parametri(separabile):.2f}x")

teorico = (K * K * C_OUT) / (K * K + C_OUT)
print(f"previsto dalla formula: {teorico:.2f}x   (limite: {K * K}x)")

## Una rete, molti compiti: l'apprendimento multi-compito

[Leggi la pagina](https://book.paithon.it/main/DeepLearning/multi-compito.html)


### In pratica: il guadagno si misura, e può essere negativo


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_num_threads(1)   # su una macchina carica i thread si ostacolano

D, H = 12, 64
N_ETICHETTATI, N_AUSILIARI, N_TEST = 40, 800, 2000   # poche etichette dove servono

def dati(seme):
    g = torch.Generator().manual_seed(seme)
    n = N_AUSILIARI + N_TEST
    X = torch.randn(n, D, generator=g)
    w = torch.randn(D, generator=g)
    nascosto = torch.tanh(X @ w)                 # la quantità che conta davvero
    return X, {
        "principale": nascosto,                  # etichettata solo su 40 esempi
        "parente":    nascosto ** 2,             # dipende dalla STESSA quantità
        # bersaglio che non dipende da X: nessuna rete può impararlo
        "rumore":     torch.randn(n, generator=g),
    }

def addestra(X, y, ausiliario, seme, passi=800, peso=1.0):
    torch.manual_seed(seme)
    tronco = nn.Sequential(nn.Linear(D, H), nn.Tanh(), nn.Linear(H, H), nn.Tanh())
    teste = nn.ModuleDict({k: nn.Linear(H, 1) for k in y})
    ott = torch.optim.Adam(list(tronco.parameters()) + list(teste.parameters()),
                           lr=3e-3)
    for _ in range(passi):
        # il compito principale vede 40 esempi, l'ausiliario ne vede 800
        perdita = F.mse_loss(teste["principale"](tronco(X[:N_ETICHETTATI])).squeeze(-1),
                             y["principale"][:N_ETICHETTATI])
        if ausiliario:
            perdita = perdita + peso * F.mse_loss(
                teste[ausiliario](tronco(X[:N_AUSILIARI])).squeeze(-1),
                y[ausiliario][:N_AUSILIARI])
        ott.zero_grad(); perdita.backward(); ott.step()
    with torch.no_grad():
        pred = teste["principale"](tronco(X[N_AUSILIARI:])).squeeze(-1)
        return F.mse_loss(pred, y["principale"][N_AUSILIARI:]).item()

def prova(ausiliario, peso=1.0):
    errori = [addestra(*dati(s)[:2], ausiliario, s, peso=peso) for s in range(5)]
    return sum(errori) / len(errori)

# cinque semi per configurazione: qualche minuto di attesa
base = prova(None)
print(f"ausiliario: {'nessuno':<14} errore sul test {base:.4f}   (  +0%)")
for ausiliario, peso in (("parente", 1.0), ("rumore", 1.0),
                         ("rumore", 0.1), ("rumore", 0.01)):
    media = prova(ausiliario, peso)
    nome = ausiliario if peso == 1.0 else f"{ausiliario} x{peso}"
    print(f"ausiliario: {nome:<14} errore sul test {media:.4f}"
          f"   ({100 * (media - base) / base:+4.0f}%)")

## Imparare a imparare in fretta

[Leggi la pagina](https://book.paithon.it/main/DeepLearning/meta-apprendimento.html)


### In pratica: dieci punti su un'onda mai vista


In [ ]:
import torch

torch.set_num_threads(1)   # su una macchina carica i thread si ostacolano

def pesi(gen, misure=((1, 40), (40, 40), (40, 1))):
    """La rete come lista esplicita di tensori: serve perche' il ciclo interno
    deve produrre una lista NUOVA di parametri, senza toccare quella vecchia."""
    p = []
    for entra, esce in misure:
        w = torch.randn(entra, esce, generator=gen) * (2.0 / entra) ** 0.5
        p += [w.requires_grad_(), torch.zeros(esce, requires_grad=True)]
    return p

def rete(x, p):
    h = torch.relu(x @ p[0] + p[1])
    h = torch.relu(h @ p[2] + p[3])
    return h @ p[4] + p[5]

def compito(gen):
    """Un membro della famiglia: ampiezza e fase sorteggiate."""
    A = torch.rand(1, generator=gen) * 4.9 + 0.1
    fase = torch.rand(1, generator=gen) * torch.pi
    return lambda x: A * torch.sin(x + fase)

def punti(f, n, gen):
    x = torch.rand(n, 1, generator=gen) * 10 - 5
    return x, f(x)

def adatta(p, x, y, passi, alfa, grafo):
    """Il ciclo interno. Con grafo=True la catena resta derivabile, ed e' cio'
    che permette al ciclo esterno di derivare ATTRAVERSO l'adattamento."""
    for _ in range(passi):
        perdita = ((rete(x, p) - y) ** 2).mean()
        g = torch.autograd.grad(perdita, p, create_graph=grafo)
        p = [w - alfa * gw for w, gw in zip(p, g)]
    return p

ITER, LOTTO = 1000, 8

# --- meta-addestramento: si valuta il DOPO, non l'adesso
gen = torch.Generator().manual_seed(1)
maml = pesi(gen)
opt = torch.optim.Adam(maml, lr=1e-3)
for _ in range(ITER):
    perdita = 0.0
    for _ in range(LOTTO):
        f = compito(gen)
        xs, ys = punti(f, 10, gen)      # insieme di supporto
        xq, yq = punti(f, 10, gen)      # insieme di interrogazione
        adattati = adatta(maml, xs, ys, 1, 0.01, grafo=True)
        perdita = perdita + ((rete(xq, adattati) - yq) ** 2).mean()
    opt.zero_grad(); (perdita / LOTTO).backward(); opt.step()

# --- il termine di paragone: la stessa rete allenata su TUTTE le sinusoidi
gen2 = torch.Generator().manual_seed(1)
insieme = pesi(gen2)
opt2 = torch.optim.Adam(insieme, lr=1e-3)
for _ in range(ITER):
    perdita = 0.0
    for _ in range(LOTTO):
        f = compito(gen2)
        x, y = punti(f, 20, gen2)
        perdita = perdita + ((rete(x, insieme) - y) ** 2).mean()
    opt2.zero_grad(); (perdita / LOTTO).backward(); opt2.step()

# --- la prova: 100 sinusoidi mai viste, stesso adattamento per tutti e tre
import statistics
prova = torch.linspace(-5, 5, 200).reshape(-1, 1)
righe = {}
for etichetta, p0 in (("a caso", pesi(torch.Generator().manual_seed(3))),
                      ("allenata su tutte", insieme),
                      ("MAML", maml)):
    g = torch.Generator().manual_seed(7)      # le stesse 100 sinusoidi per tutti
    prima, dopo = [], []
    for _ in range(100):
        f = compito(g)
        xs, ys = punti(f, 10, g)
        with torch.no_grad():
            prima.append(((rete(prova, p0) - f(prova)) ** 2).mean().item())
        p1 = adatta([w.detach().requires_grad_() for w in p0],
                    xs, ys, 5, 0.01, grafo=False)
        with torch.no_grad():
            dopo.append(((rete(prova, p1) - f(prova)) ** 2).mean().item())
    righe[etichetta] = (statistics.median(prima), statistics.median(dopo),
                        sum(1 for a, b in zip(prima, dopo) if b < a))

print("errore quadratico mediano su 100 sinusoidi mai viste")
print("(mediano e non medio: una singola divergenza rende la media inutile)")
print(f"   {'':20s} {'prima':>8s} {'dopo 5 passi':>13s}   migliora in")
for etichetta, (a, b, quante) in righe.items():
    print(f"   {etichetta:20s} {a:8.2f} {b:13.2f}   {quante:3d} casi su 100")

with torch.no_grad():
    u = rete(prova, insieme)
    print(f"\nla rete allenata su tutte oscilla fra {u.min():.2f} e {u.max():.2f}:")
    print("e' la media della famiglia: un'onda sola, di ampiezza ridotta")

### Confrontare invece di adattare


In [ ]:
import numpy as np

rng = np.random.default_rng(0)
d, r = 40, 4                   # quaranta misure, e contano quattro direzioni
# le direzioni che contano, e quelle di un'altra famiglia
famiglia = np.linalg.qr(rng.normal(size=(d, r)))[0]
altra = np.linalg.qr(rng.normal(size=(d, r)))[0]

def classi(n, direzioni):
    """n centri di classe, sparsi soltanto lungo le direzioni date."""
    return rng.normal(0, 3, (n, r)) @ direzioni.T

def episodio(centri, k, q=5, N=5):
    """N classi a caso, k esempi di supporto e q domande per ciascuna."""
    scelte = centri[rng.choice(len(centri), N, replace=False)]
    # il rumore sta su tutte le misure
    supporto = scelte[:, None] + rng.normal(0, 1, (N, k, d))
    domande = scelte[:, None] + rng.normal(0, 1, (N, q, d))
    return supporto, domande.reshape(-1, d), np.repeat(np.arange(N), q)

def accuratezza(W, centri, k, episodi=1000):
    giuste = totale = 0
    for _ in range(episodi):
        S, Q, y = episodio(centri, k)
        prototipi = S.mean(1) @ W.T            # nello spazio appreso
        dist = (((Q @ W.T)[:, None] - prototipi[None]) ** 2).sum(-1)
        giuste, totale = giuste + (dist.argmin(1) == y).sum(), totale + len(y)
    return giuste / totale

base = classi(64, famiglia)                    # le classi dell'addestramento
W = np.eye(d)                     # si parte dalle misure così come sono
for _ in range(2000):             # addestramento a episodi, 5-way 1-shot
    S, Q, y = episodio(base, 1)
    U = Q[:, None, :] - S.mean(1)[None]        # domanda meno prototipo
    logit = -((U @ W.T) ** 2).sum(-1)
    p = np.exp(logit - logit.max(1, keepdims=True))
    p /= p.sum(1, keepdims=True)
    p[np.arange(len(y)), y] -= 1               # g: probabilità meno indicatrice
    M = np.einsum("qk,qki,qkj->ij", p / len(y), U, U)
    W += 0.01 * 2 * W @ M                      # il gradiente è -2 W M

nuove, estranee = classi(64, famiglia), classi(64, altra)
for nome, centri in [("classi nuove, stessa famiglia", nuove),
                     ("classi di un'altra famiglia", estranee)]:
    for k in (1, 5):
        grezze = accuratezza(np.eye(d), centri, k)
        apprese = accuratezza(W, centri, k)
        print(f"{nome}, {k}-shot: misure grezze {grezze:.1%},"
              f" spazio appreso {apprese:.1%}")
resto = np.linalg.svd(famiglia, full_matrices=True)[0][:, r:]   # le altre 36
dentro = np.linalg.norm(W @ famiglia) / np.sqrt(r)
fuori = np.linalg.norm(W @ resto) / np.sqrt(d - r)
print(f"fattore medio: {dentro:.2f} sulle 4 direzioni che contano,"
      f" {fuori:.2f} sulle altre 36")